# Exercise 2: Creating and Populating Graphs in Neo4j - Design Patterns and Refactoring

In this exercise, we move beyond basic graph creation and explore **how query patterns influence your graph design**. Unlike relational databases where the schema is fixed before application development, graph database design is iterative and driven by the questions you want to answer.

Recall that graph databases treat **connections between entities as first-class data**. Relationships are stored directly — they have direction, type, and can carry properties of their own. This is fundamentally different from relational databases (where relationships are implied via JOINs) or document databases (where relationships are handled through embedding or manual lookups).

Key concepts we will review include:
1. **Graph Data Modeling**: Understanding nodes, relationships, and properties
2. **Design Patterns**: Common patterns for different use cases (simple, hub nodes, intermediate nodes, denormalized, and batch patterns)
3. **Query-Driven Design**: How your queries should shape your model — graph query performance is proportional to traversal length, not table size
4. **Normalization vs. Denormalization**: When to split data into more nodes vs. collapsing information into properties
5. **Refactoring**: Improving your graph as requirements change — graphs naturally extend without schema migrations
6. **Relationship Types**: Why specifying descriptive relationship types matters for readability and maintainability

By the end of this exercise, you'll understand how to design scalable graph databases that perform efficiently for your specific use cases.

## Step 1: Setup and Connection

Let's establish our connection to Neo4j and prepare for a deeper dive into graph modeling.

In [ ]:
%%capture
# Install required packages
%pip install neo4j networkx matplotlib pandas numpy

# Import required libraries
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
import time
import pandas as pd
import json
from datetime import datetime, timedelta
import random

# Driver setup - connect to Neo4j instance
URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "password")

# Create driver
driver = GraphDatabase.driver(URI, auth=AUTH)

# Test connection
try:
    with driver.session() as session:
        session.run("RETURN 1")
    print("Successfully connected to Neo4j")
except ServiceUnavailable:
    print("Could not connect to Neo4j. Make sure the service is running.")
    driver.close()

## Step 2: Understanding Graph Data Model Design Principles

Before we build a graph, let's understand the key design principles.

First, let's consider the key elements of a graph: **Nodes, Relationships, and Properties**
- **Nodes**: Represent entities — the "things in the real world" you want to query, aggregate, or connect (analogous to rows in relational databases or documents in MongoDB).
- **Relationships**: The connections between those nodes — how they interact and are related to one another. Unlike relational databases where relationships are implied and reconstructed via JOINs, graph databases store relationships directly as first-class data objects.
- **Properties**: Descriptive attributes stored on both nodes and relationships (analogous to columns in SQL or fields in a document).
- **Labels**: Classifications that describe the type of node (e.g., `Customer`, `Product`). Unlike tables or collections, labels don't determine physical storage — they're just a way to describe what kind of thing a node represents.

When working with nodes, relationships, and properties, be explicit: use descriptive relationship types (not generic "HAS" or "IS"). This helps make queries more human-readable: `(customer)-[:PURCHASED]->(product)` is better than `(customer)-[:RELATED]->(product)`.

Next, when you design your graph, ask yourself: "What questions do I want to answer?". This will help you structure your graph to make those queries efficient. The right model depends on your access patterns, not universal rules. Follow these modeling steps:

**1) Focus on the application you're building**
- Write down the key questions the system needs to answer
- What nodes are those questions about? What properties do you need?
- What relationships connect them — and what properties belong on those relationships?

**2) Prioritize representative queries**
- Sketch out a few representative queries and prioritize them
- In graph databases, query performance is proportional to the **traversal length** (number of hops), not the size of the dataset — so you want the most common paths to be efficient

**3) Determine the level of normalization**
- **Normalize** (more nodes/relationships) when: data has high cardinality, information would be duplicated across many nodes, or an entity makes sense on its own and is queried independently
- **Denormalize** (collapse into properties) when: it simplifies the model, data is always accessed together, or further normalization would significantly increase hops for common traversals
- If further normalization significantly increases the number of hops for your most common queries, that's a strong signal to denormalize instead


Depending on the answers to the above investigations, you may consider some **Common Design Patterns**:
- **Simple Patterns**: With central nodes connected directly to many others (e.g., `Customer -[:PURCHASED]-> Product`). Best for straightforward queries with few entity types.
- **Highly Normalized Graphs**: Introduce additional nodes and relationships to avoid duplication — splitting data into more granular nodes (e.g., adding `Category` or `CustomerSegment` hub nodes) so that aggregation and filtering queries become efficient.
- **Intermediate Nodes**: Create explicit nodes for junction objects that represent events or transactions (e.g., `Order` between `Customer` and `Product`, or `Batch` nodes for tracking shipments in sequence). This elevates a relationship into a first-class entity with its own properties and connections.
- **Denormalized Graphs**: Collapse information into node or relationship properties to reduce traversal hops, even if it means some duplication. Best when data is always accessed together and the number of hops for common queries matters most.
- **Batch Patterns**: Model ordered sequences of events using intermediate nodes linked in a chain (e.g., `Batch` nodes connected by `:NEXT` relationships). Useful when you need to track temporal ordering, such as "Which batch are we currently selling from?" or "Will we run out before the next batch arrives?"

Let's explore these through a real example.


## Step 3: Preparation - Clear and Load ACME Corporation Data

We'll use the ACME Corporation dataset (customers, products, purchases, ratings) to demonstrate different graph design approaches and how to refactor when requirements change.

In [ ]:
# Clear all existing data
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("✅ Cleared all nodes and relationships")

In [ ]:
# Load sample ACME Corporation data
acme_data = {
    'customers': [
        {'id': 'C001', 'name': 'Alice Smith', 'email': 'alice@company.com', 'status': 'gold'},
        {'id': 'C002', 'name': 'Bob Johnson', 'email': 'bob@company.com', 'status': 'silver'},
        {'id': 'C003', 'name': 'Charlie Lee', 'email': 'charlie@company.com', 'status': 'bronze'},
        {'id': 'C004', 'name': 'Diana Prince', 'email': 'diana@company.com', 'status': 'gold'},
        {'id': 'C005', 'name': 'Eve Martinez', 'email': 'eve@company.com', 'status': 'silver'},
    ],
    'products': [
        {'id': 'P001', 'name': 'Small Gear', 'category': 'Gears', 'type': 'small', 'price': 49.99},
        {'id': 'P002', 'name': 'Large Gear', 'category': 'Gears', 'type': 'large', 'price': 149.99},
        {'id': 'P003', 'name': 'Bearing', 'category': 'Components', 'type': 'bearing', 'price': 29.99},
        {'id': 'P004', 'name': 'Shaft', 'category': 'Components', 'type': 'shaft', 'price': 39.99},
    ],
    'purchases': [
        {'customer_id': 'C001', 'product_id': 'P001', 'date': '2024-01-15', 'qty': 5},
        {'customer_id': 'C001', 'product_id': 'P003', 'date': '2024-01-20', 'qty': 10},
        {'customer_id': 'C002', 'product_id': 'P002', 'date': '2024-01-18', 'qty': 2},
        {'customer_id': 'C002', 'product_id': 'P004', 'date': '2024-02-01', 'qty': 8},
        {'customer_id': 'C003', 'product_id': 'P001', 'date': '2024-02-05', 'qty': 3},
        {'customer_id': 'C004', 'product_id': 'P001', 'date': '2024-02-10', 'qty': 7},
        {'customer_id': 'C004', 'product_id': 'P002', 'date': '2024-02-12', 'qty': 4},
        {'customer_id': 'C005', 'product_id': 'P003', 'date': '2024-02-15', 'qty': 6},
    ],
    'ratings': [
        {'customer_id': 'C001', 'product_id': 'P001', 'score': 5, 'text': 'Excellent quality'},
        {'customer_id': 'C001', 'product_id': 'P003', 'score': 4, 'text': 'Good product'},
        {'customer_id': 'C002', 'product_id': 'P002', 'score': 5, 'text': 'Perfect fit'},
        {'customer_id': 'C003', 'product_id': 'P001', 'score': 3, 'text': 'Average'},
        {'customer_id': 'C004', 'product_id': 'P001', 'score': 5, 'text': 'Great!'},
        {'customer_id': 'C004', 'product_id': 'P002', 'score': 4, 'text': 'Very good'},
    ]
}

print("✅ ACME Corporation data loaded:")
print(f"   Customers: {len(acme_data['customers'])}")
print(f"   Products: {len(acme_data['products'])}")
print(f"   Purchases: {len(acme_data['purchases'])}")
print(f"   Ratings: {len(acme_data['ratings'])}")

## Step 4: Design Pattern 1 - Simple Model (Basic Approach)

Let's start with a straightforward graph model:
- **Customers** connect to **Products** via PURCHASED relationships
- **Customers** rate **Products** via RATED relationships

This works well for basic queries but has limitations for complex analytics. Let's implement it and understand its strengths and weaknesses.

In [ ]:
def create_simple_model(driver, data):
    """
    Model 1: Simple direct connections
    Customers <-[PURCHASED]-> Products
    Customers <-[RATED]-> Products
    """
    ### BEGIN SOLUTION
    with driver.session() as session:
        # 1. Create Customer nodes
        for customer in data['customers']:
            session.run(
                "CREATE (c:Customer {id: $id, name: $name, email: $email, status: $status})",
                id=customer['id'], name=customer['name'],
                email=customer['email'], status=customer['status']
            )

        # 2. Create Product nodes
        for product in data['products']:
            session.run(
                "CREATE (p:Product {id: $id, name: $name, category: $category, type: $type, price: $price})",
                id=product['id'], name=product['name'],
                category=product['category'], type=product['type'], price=product['price']
            )

        # 3. Create PURCHASED relationships with date and quantity
        for purchase in data['purchases']:
            session.run("""
                MATCH (c:Customer {id: $customer_id}), (p:Product {id: $product_id})
                CREATE (c)-[:PURCHASED {date: $date, quantity: $qty}]->(p)
            """, customer_id=purchase['customer_id'], product_id=purchase['product_id'],
                date=purchase['date'], qty=purchase['qty'])

        # 4. Create RATED relationships with score and text
        for rating in data['ratings']:
            session.run("""
                MATCH (c:Customer {id: $customer_id}), (p:Product {id: $product_id})
                CREATE (c)-[:RATED {score: $score, text: $text}]->(p)
            """, customer_id=rating['customer_id'], product_id=rating['product_id'],
                score=rating['score'], text=rating['text'])

        # Print summary
        result = session.run("MATCH (n) RETURN labels(n)[0] AS type, count(*) AS count ORDER BY type")
        print("✅ Simple model created:")
        for record in result:
            print(f"   {record['type']}: {record['count']}")

        result = session.run("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS count ORDER BY type")
        print("   Relationships:")
        for record in result:
            print(f"   {record['type']}: {record['count']}")
    ### END SOLUTION

# Create the simple model
create_simple_model(driver, acme_data)

### Analysis of Simple Model

This is the most basic example of a graph — a **denormalized** approach where we collapse information directly onto nodes and relationship properties rather than splitting it into separate entities.

**Strengths:**
- Simple to understand and implement
- Great for direct traversal queries like "Who purchased what?" or "What did Alice rate?"
- Relationship properties (date, quantity, score) keep related data together — no extra hops needed
- Fewer nodes means a smaller graph to manage

This maps to the **denormalization** guidance from the course: if data is always accessed together (e.g., purchase date and quantity), storing it on the relationship avoids coordination overhead and reduces traversal length.

**Weaknesses:**
- Hard to analyze trends by category — you'd need to filter by product properties rather than traversing to a category node
- Difficult to find products bought together frequently
- No explicit representation of categories or customer segments as queryable entities
- Can't efficiently aggregate by category without scanning all products
- If you need to query or aggregate by a concept (like "Gears" category), it should be its own node — not just a property

This is a classic case where the **"Properties vs. Nodes" decision** matters: if you'll query or aggregate on something, make it a node.

## Step 5: Design Pattern 2 - Enhanced Model (Hub Node Pattern / Normalization)

Now let's refactor by adding **Category** nodes and **CustomerSegment** nodes. This is the **Hub Node Pattern** — a form of **graph normalization** where we introduce additional nodes and relationships instead of repeating properties.

In graph databases, normalization means splitting data into more granular nodes so that aggregation and filtering queries become efficient. This is analogous to how normalization works in relational databases (splitting data across tables to avoid duplication), but here we're creating **hub nodes** — categorical nodes that serve as shared connection points.

**Why this matters for queries:**
- "Show me top products in the Gears category" → direct traversal to the Category hub node
- "Find gold customers" → query by the CustomerSegment hub node
- "What categories interest gold customers most?" → efficient pattern matching through hub nodes
- Without hub nodes, these queries would require filtering by property values across all Product or Customer nodes

In [ ]:
def create_enhanced_model(driver, data):
    """
    Model 2: Enhanced model with hub nodes
    Categories and CustomerSegments serve as "hub" nodes
    Products -[:BELONGS_TO]-> Category
    Customers -[:HAS_SEGMENT]-> CustomerSegment
    """
    ### BEGIN SOLUTION
    with driver.session() as session:
        # 1. Extract unique categories from products
        categories = set(p['category'] for p in data['products'])

        # 2. Extract unique customer segments (status values) from customers
        segments = set(c['status'] for c in data['customers'])

        # 3. Create Category hub nodes
        for category in categories:
            session.run("CREATE (c:Category {name: $name})", name=category)

        # 4. Create CustomerSegment hub nodes
        for segment in segments:
            session.run("CREATE (s:CustomerSegment {name: $name})", name=segment)

        # 5. Link Products to Categories with BELONGS_TO relationships
        for product in data['products']:
            session.run("""
                MATCH (p:Product {id: $id}), (c:Category {name: $category})
                CREATE (p)-[:BELONGS_TO]->(c)
            """, id=product['id'], category=product['category'])

        # 6. Link Customers to CustomerSegments with HAS_SEGMENT relationships
        for customer in data['customers']:
            session.run("""
                MATCH (c:Customer {id: $id}), (s:CustomerSegment {name: $status})
                CREATE (c)-[:HAS_SEGMENT]->(s)
            """, id=customer['id'], status=customer['status'])

        print("✅ Enhanced model with hub nodes created")
        print(f"   Categories: {categories}")
        print(f"   Segments: {segments}")
    ### END SOLUTION

# Add enhanced model on top of existing simple model
create_enhanced_model(driver, acme_data)

### Analysis of Enhanced Model with Hub Nodes

By normalizing our graph with hub nodes, we've made a key trade-off: slightly more structure in exchange for significantly more powerful queries.

**New Capabilities (Normalization Benefits):**
- Can now efficiently query "What are the top-rated products in category X?" — start at the Category node and traverse directly
- Can find "Which categories do gold customers prefer?" — traverse from CustomerSegment → Customers → Products → Categories
- Hub nodes make aggregation queries much faster because they act as shared connection points
- Clear separation of concerns — entities that deserve independent querying (categories, segments) now have their own nodes
- Reduces risk of inconsistency: category names live in one place rather than being duplicated as properties across many Product nodes

**Trade-offs (Normalization Costs):**
- Slightly more complex graph structure with additional node types
- More nodes and relationships to maintain
- Additional hops required for some traversals (e.g., getting a product's category now requires one extra hop)

This is a good example of the normalization guidance from the course: **if the same information would be duplicated across many nodes, normalization reduces the risk of inconsistency**. And **if an entity makes sense on its own and is queried independently, it likely deserves its own place in the graph**.

## Step 6: Intermediate Nodes — Elevating Relationships to First-Class Entities

One of the powerful features of graphs is the ability to store data on relationships (rich relationships), not just nodes. But sometimes a relationship carries so much information that it deserves to become its own **intermediate node**.

An intermediate node is created when a relationship has enough properties and connections that it starts being its own record — it has properties and connects to other nodes independently.

**When to use intermediate nodes:**
- When an event or transaction (like an Order) needs to connect to multiple other entities
- When you need to track status, fulfillment, or other evolving properties on what was originally "just a connection"
- When the same concept appears in ordered sequences (like Batches arriving over time)

**The pattern shift:**
- **Before**: `Customer -[:PURCHASED {quantity, date}]-> Product` (data lives on the relationship)
- **After**: `Customer -[:PLACED]-> Order -[:CONTAINS]-> Product` (Order becomes a first-class entity)

This is particularly useful for **Batch Patterns** as well — when real-world business processes are ordered in sequence (e.g., shipments arriving in batches), intermediate batch nodes linked in a chain can answer questions like "Which batch are we currently selling from?" or "Will stock run out before the next batch arrives?"

In [ ]:
def add_order_nodes(driver, data):
    """
    Design Pattern 3: Intermediate Nodes (Orders)
    Instead of: Customer -[PURCHASED {quantity, date}]-> Product
    Use: Customer -[PLACED]-> Order <-[CONTAINS]- Product
    
    Benefits:
    - Order becomes a first-class entity (can have relationships to other things)
    - Can track order status, fulfillment, payments
    - Can query "which orders contain multiple products?"
    """
    ### BEGIN SOLUTION
    with driver.session() as session:
        for i, purchase in enumerate(data['purchases'], 1):
            order_id = f"ORD{i:04d}"
            session.run("""
                MATCH (c:Customer {id: $customer_id}), (p:Product {id: $product_id})
                CREATE (o:Order {id: $order_id, date: $date})
                CREATE (c)-[:PLACED]->(o)
                CREATE (o)-[:CONTAINS {quantity: $qty}]->(p)
            """, customer_id=purchase['customer_id'], product_id=purchase['product_id'],
                order_id=order_id, date=purchase['date'], qty=purchase['qty'])

        print(f"✅ Created {len(data['purchases'])} Order intermediate nodes")
        result = session.run("MATCH (o:Order) RETURN count(o) AS count")
        print(f"   Total Orders in graph: {result.single()['count']}")
    ### END SOLUTION

# Add Order intermediate nodes
add_order_nodes(driver, acme_data)

### Intermediate Node Benefits

By promoting `Order` from a relationship to a node, we've unlocked sophisticated query patterns:
- "Find customers who ordered multiple products in a single order"
- "Which products are frequently ordered together?" (co-occurrence within orders)
- "Find the most recent order from each customer"
- "Track order status and fulfillment separately from the original purchase"

**Why this matters:** The Order node can now have its own relationships to other entities — payments, shipments, returns — without overloading a single relationship with too many properties. As the course teaches, when a relationship has enough information that it "starts being its own record," it deserves to be an intermediate node.

**Batch extension:** If orders arrived in batches, we could further add `Batch` intermediate nodes chained with `:NEXT` relationships to track temporal ordering — enabling questions like "Which batch did a customer's order come from?" or "Will we run out of stock before the next batch arrives?"

## Step 7: Demonstrating Query-Driven Design

Now let's show why our design choices matter by querying the enhanced model.

#### Task 1: Query by Category Hub

Find all products in a category and their average rating.

In [ ]:
# Efficient category query - hub pattern makes this fast
### BEGIN SOLUTION
category_query = """
MATCH (cat:Category {name: 'Gears'})<-[:BELONGS_TO]-(p:Product)
OPTIONAL MATCH (p)<-[r:RATED]-()
RETURN p.name AS product, p.price AS price,
       avg(r.score) AS avg_rating, count(r) AS num_reviews
ORDER BY avg_rating DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(category_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=records[0].keys())
        print("Products in 'Gears' Category with Ratings:")
        print(df)
    else:
        print("No results - check your query")

#### Task 2: Customer Segment Analysis

Find the purchasing patterns of different customer segments.

In [ ]:
# Find which categories each segment purchases from
### BEGIN SOLUTION
segment_analysis_query = """
MATCH (seg:CustomerSegment {name: 'gold'})<-[:HAS_SEGMENT]-(c:Customer)
      -[:PLACED]->(o:Order)-[cont:CONTAINS]->(p:Product)
      -[:BELONGS_TO]->(cat:Category)
RETURN cat.name AS category,
       count(DISTINCT c) AS unique_customers,
       sum(cont.quantity) AS total_quantity
ORDER BY total_quantity DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(segment_analysis_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=records[0].keys())
        print("Gold Customer Segment Analysis:")
        print(df)
    else:
        print("No results - check your query")

#### Task 3: Product Affinity (Products Bought Together)

Find products that are frequently purchased together using the Order intermediate node pattern.

In [ ]:
# Find products purchased by the same customers
### BEGIN SOLUTION
affinity_query = """
MATCH (c:Customer)-[:PURCHASED]->(p1:Product),
      (c)-[:PURCHASED]->(p2:Product)
WHERE p1.id < p2.id
RETURN p1.name AS product1, p2.name AS product2,
       count(DISTINCT c) AS shared_customers
ORDER BY shared_customers DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(affinity_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=records[0].keys())
        print("Product Affinity Analysis (Bought Together):")
        print(df)
    else:
        print("No co-purchases found")

#### Task 4: Customer Lifetime Value

Calculate total spending and order count per customer using the Order and Product relationship structure.

In [ ]:
# Customer Lifetime Value calculation
### BEGIN SOLUTION
clv_query = """
MATCH (c:Customer)-[:HAS_SEGMENT]->(seg:CustomerSegment)
MATCH (c)-[:PLACED]->(o:Order)-[cont:CONTAINS]->(p:Product)
RETURN c.name AS customer, seg.name AS segment,
       count(DISTINCT o) AS num_orders,
       sum(p.price * cont.quantity) AS total_spent
ORDER BY total_spent DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(clv_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=records[0].keys())
        print("Customer Lifetime Value Analysis:")
        print(df)
    else:
        print("No results")

## Step 8: Refactoring — Adding New Requirements

Real-world systems evolve. One of the key advantages of graph databases over relational systems is that **structure changes happen through regular graph operations, not a separate schema migration step**. There's no need to run an `ALTER TABLE`-style command — you simply add new nodes, relationships, and properties as requirements emerge.

Graph databases use a **flexible data model**: nodes and relationships can have properties added, changed, or removed at any time, and different nodes of the same type don't need to share the same structure. Structure emerges as data is created and updated.

Let's demonstrate this by adding a new requirement: **Suppliers** — products have suppliers, and we want to analyze supplier relationships. We'll add `Supplier` nodes and `SUPPLIED_BY` relationships without modifying any existing data.

In [ ]:
def add_supplier_network(driver):
    """
    Refactoring: Add Supplier nodes and relationships
    This reduces duplication and allows supplier-based analytics
    """
    ### BEGIN SOLUTION
    suppliers = [
        {'id': 'S001', 'name': 'GearWorks Inc.'},
        {'id': 'S002', 'name': 'ComponentCo Ltd.'},
    ]

    product_suppliers = {
        'P001': 'S001', 'P002': 'S001',  # Gears from GearWorks
        'P003': 'S002', 'P004': 'S002',  # Components from ComponentCo
    }

    with driver.session() as session:
        # Create Supplier nodes
        for supplier in suppliers:
            session.run(
                "CREATE (s:Supplier {id: $id, name: $name})",
                id=supplier['id'], name=supplier['name']
            )

        # Link Products to Suppliers with SUPPLIED_BY relationships
        for product_id, supplier_id in product_suppliers.items():
            session.run("""
                MATCH (p:Product {id: $pid}), (s:Supplier {id: $sid})
                CREATE (p)-[:SUPPLIED_BY]->(s)
            """, pid=product_id, sid=supplier_id)

        print("✅ Supplier network added")
        print(f"   Suppliers: {[s['name'] for s in suppliers]}")
    ### END SOLUTION

# Add supplier network
add_supplier_network(driver)

#### Task 5: Supplier Performance Analysis

Now we can analyze supplier performance using our refactored graph.

In [ ]:
# Supplier performance query
### BEGIN SOLUTION
supplier_performance_query = """
MATCH (s:Supplier)<-[:SUPPLIED_BY]-(p:Product)
OPTIONAL MATCH (o:Order)-[cont:CONTAINS]->(p)
OPTIONAL MATCH ()-[r:RATED]->(p)
RETURN s.name AS supplier,
       count(DISTINCT p) AS product_count,
       count(DISTINCT o) AS order_count,
       avg(r.score) AS avg_rating,
       sum(cont.quantity) AS total_units_sold
ORDER BY total_units_sold DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(supplier_performance_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=records[0].keys())
        print("Supplier Performance Analysis:")
        print(df)
    else:
        print("No results")

## Step 9: Testing Relationship Type Importance

Let's demonstrate why specifying relationship types matters. Relationship types should be **verb phrases** that describe the action or connection between nodes.

Recall from the course: in graph databases, relationships are first-class data objects — they are stored directly, have direction and type, and can carry properties. Because graph queries use pattern matching (e.g., `MATCH (c)-[:PURCHASED]->(p)`), the relationship type name directly impacts both **readability** (can someone understand what the query does?) and **maintainability** (can someone extend or debug it later?).

Bad naming makes queries ambiguous and the graph harder to evolve. Good naming makes the graph self-documenting.

In [ ]:
# Example comparison
print("❌ BAD RELATIONSHIP NAMING:")
print("   Customer -[HAS]-> Product")
print("   Question: Does the customer own the product? Have they purchased it? Are they interested in it?")
print("   -> Ambiguous and hard to maintain\n")

print("✅ GOOD RELATIONSHIP NAMING:")
print("   Customer -[PURCHASED]-> Product  (indicates transaction)")
print("   Customer -[RATED]-> Product      (indicates review)")
print("   Customer -[INTERESTED_IN]-> Product (indicates browsing/wish list)")
print("   -> Clear intent, self-documenting, maintainable\n")

print("BEST PRACTICE: Relationship types should be VERB phrases describing the action or connection!")

## Step 10: Summary - Design Patterns Comparison

Let's visualize and compare the different models we've created.

In [ ]:
# Summary statistics of our final graph
### BEGIN SOLUTION
stats_query = """
MATCH (n)
RETURN labels(n)[0] AS node_type, count(*) AS count
ORDER BY node_type
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(stats_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=['Node Type', 'Count'])
        print("Final Graph Statistics:")
        print(df)
        print(f"\nTotal nodes: {df['Count'].sum()}")
    else:
        print("No results")

# Relationship statistics
### BEGIN SOLUTION
rel_stats_query = """
MATCH ()-[r]->()
RETURN type(r) AS rel_type, count(*) AS count
ORDER BY count DESC
"""
### END SOLUTION

with driver.session() as session:
    result = session.run(rel_stats_query)
    records = list(result)
    if records:
        df = pd.DataFrame([record.values() for record in records], columns=['Relationship Type', 'Count'])
        print("\nRelationship Statistics:")
        print(df)
        print(f"Total relationships: {df['Count'].sum()}")
    else:
        print("No results")

## Summary: Key Lessons in Graph Data Modeling

### 1. **Query-Driven Design**
Your graph structure should match your query patterns. There's no universal "best" model — only the best model for your use case. In graph databases, query performance is proportional to **traversal length** (number of hops), not dataset size. So you want the most common paths to be efficient.

### 2. **Hub Node Pattern (Normalization)**
Create categorical/dimensional nodes (Categories, Segments, Suppliers) to enable efficient aggregation and filtering queries. This is graph normalization — introducing additional nodes and relationships instead of repeating properties across many nodes.

### 3. **Intermediate Nodes**
When a relationship carries enough information that it "starts being its own record," promote it to a node. Orders, Batches, Reviews, and Shipments are common examples. Intermediate nodes can have their own properties and connect to other entities independently.

### 4. **Rich Relationships (Denormalization)**
Store important metadata directly on relationships as properties when the data is always accessed together and keeping it on the relationship avoids unnecessary hops. This is graph denormalization — collapsing information into properties for performance.

### 5. **Relationship Type Naming**
Use verb phrases for relationship types (PURCHASED, RATED, SUPPLIED_BY). Avoid generic names like HAS or IS. Relationships are first-class data objects in graph databases — they have direction, type, and properties — so clear naming makes the graph self-documenting and maintainable.

### 6. **Refactoring**
As requirements change, add new node types and relationships through regular graph operations. Graph databases use a flexible data model — structure emerges and evolves as data is created and updated, without `ALTER TABLE`-style schema migrations.

### 7. **Normalization vs. Denormalization Trade-offs**

| **Denormalize when...** | **Normalize when...** |
|---|---|
| It simplifies the model and reasoning | Data has high or growing cardinality |
| Data is always accessed together | Same information would be duplicated across many nodes |
| Data is updated together | An entity makes sense on its own and is queried independently |
| Fewer hops for common traversals | You need to reduce risk of inconsistency |

**Critical signal:** If further normalization significantly increases the number of hops for your most common traversals, that's a strong signal to denormalize instead.

### 8. **Design Questions to Ask**
- "What are my primary queries?" → Determines overall graph structure
- "What entities need independent querying?" → Those should be nodes, not properties
- "Are there categorical groupings I need?" → Use hub nodes
- "Can relationships carry important metadata?" → Store on relationships (denormalize)
- "What intermediate objects represent events or transactions?" → Create intermediate nodes
- "What is the traversal length for my most common queries?" → Optimize for shortest paths

### 9. **Consistency and Validation**
Graph databases are flexible by default — they accept nodes with missing properties, relationships without metadata, and varying structures across nodes of the same label. You can optionally enforce consistency using **constraints** (uniqueness, required properties) so that invalid writes are rejected at runtime. Validation is intentional in graph databases — you decide how much structure to enforce.

By designing with these patterns in mind, you create graphs that are both performant and maintainable as your application evolves!